# 02_specialization — one head field, N extension tables

A record row continues in one of several tables, and which one depends on a value the row itself carries: a first pressing, a re-pressing or a test pressing, each with its own columns. A relation as chapter 21 describes it cannot say this — a field's target type is exactly one.

A specialization declares it: the field's type is parameterised by the **union** of the alternatives, and a `Classifier` marker beside it names the choosing field and the codes.


In [ ]:
!pip install aoa-action-machine


In [ ]:
from __future__ import annotations

from typing import Annotated, Literal

from pydantic import Field, ValidationError

from aoa.action_machine.domain import (
    BaseEntity,
    Classifier,
    Generalization,
    Inverse,
    Rel,
    Specialization,
)
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.entity.entity_intent_resolver import EntityIntentResolver
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine


## The declaration

Every extension declares the way back and its own single code. The head names the choosing field and the whole code set.


In [ ]:
class MusicDomain(BaseDomain):
    name = "music"
    description = "A music library"


@entity(description="First pressing", domain=MusicDomain)
class FirstPressEntity(BaseEntity):
    id: str = Field(description="Same id as the record")
    stamper: str = Field(description="Stamper code")
    year: int = Field(description="Year of the first press")

    record: Annotated[
        Generalization[VinylRecordEntity],
        Classifier("record", Literal["first"]),
        Inverse(VinylRecordEntity, "pressing"),
    ] = Rel(description="Record this pressing belongs to")


@entity(description="Re-pressing", domain=MusicDomain)
class RepressEntity(BaseEntity):
    id: str = Field(description="Same id as the record")
    year: int = Field(description="Year of the re-press")
    remastered: bool = Field(description="Whether the audio was remastered")

    record: Annotated[
        Generalization[VinylRecordEntity],
        Classifier("record", Literal["repress"]),
        Inverse(VinylRecordEntity, "pressing"),
    ] = Rel(description="Record this pressing belongs to")


@entity(description="Test pressing", domain=MusicDomain)
class TestPressEntity(BaseEntity):
    id: str = Field(description="Same id as the record")
    approved_by: str = Field(description="Who signed it off")

    record: Annotated[
        Generalization[VinylRecordEntity],
        Classifier("record", Literal["test"]),
        Inverse(VinylRecordEntity, "pressing"),
    ] = Rel(description="Record this pressing belongs to")


@entity(description="Vinyl record", domain=MusicDomain)
class VinylRecordEntity(BaseEntity):
    id: str = Field(description="Record id")
    title: str = Field(description="Album title")
    media: str = Field(description="Classifier: which pressing this is")

    pressing: Annotated[
        Specialization[FirstPressEntity | RepressEntity | TestPressEntity],
        Classifier(field="media", codes=Literal["first", "repress", "test"]),
        Inverse(FirstPressEntity, "record"),
    ] = Rel(description="How this record was pressed")


for _cls in (FirstPressEntity, RepressEntity, TestPressEntity, VinylRecordEntity):
    _cls.model_rebuild()


## 1. What the framework reads off the head

Not a hand-read annotation: the parser the graph and the build rules consume.


In [ ]:
axis = EntityIntentResolver.resolve_entity_specializations(VinylRecordEntity)[0]
print("field        :", axis.field_name)
print("chosen by    :", axis.classifier_field)
print("alternatives :", [cls.__name__ for cls in axis.alternatives])
print("codes        :", list(axis.codes))
print("code -> class:", {code: cls.__name__ for code, cls in axis.code_to_target.items()})
print("paired field :", axis.inverse_field)


## 2. A hydrated link, and 3. another alternative in the same field

The value carries the identifier, which variant it is, and the row when one was loaded.


In [ ]:
first = FirstPressEntity(id="rec-1", stamper="1A", year=1959)
record = VinylRecordEntity(
    id="rec-1", title="Kind of Blue", media="first",
    pressing=Specialization[FirstPressEntity | RepressEntity | TestPressEntity](
        id="rec-1", variant="first", entity=first,
    ),
)
print("pressing        =", record.pressing)
print(".variant        =", record.pressing.variant)
print(".entity         =", type(record.pressing.entity).__name__)
print(".entity.stamper =", record.pressing.entity.stamper)

repress = RepressEntity(id="rec-2", year=1997, remastered=True)
other = VinylRecordEntity(
    id="rec-2", title="Kind of Blue", media="repress",
    pressing=Specialization[FirstPressEntity | RepressEntity | TestPressEntity](
        id="rec-2", variant="repress", entity=repress,
    ),
)
print()
print("another variant:", other.pressing.variant, "|", type(other.pressing.entity).__name__)


## 4. Reading which variant you got, and 5. a link with no loaded row


In [ ]:
for item in (record, other):
    linked = item.pressing.entity
    if isinstance(linked, FirstPressEntity):
        print(f"{item.id}: first press, stamper {linked.stamper}")
    elif isinstance(linked, RepressEntity):
        print(f"{item.id}: repress from {linked.year}, remastered={linked.remastered}")
    else:
        print(f"{item.id}: another pressing")

unloaded = Specialization[FirstPressEntity | RepressEntity | TestPressEntity](id="rec-3", variant="test")
print()
print("a link with no loaded row:", unloaded, "| .entity:", unloaded.entity)


## 6. What the model refuses

The type argument is what pydantic enforces: a class outside it, or a string where a row belongs.


In [ ]:
for label, value in (
    ("a string instead of a row", "not an entity"),
    ("a class outside the union", VinylRecordEntity(id="rec-9", title="Other", media="first")),
):
    try:
        Specialization[FirstPressEntity | RepressEntity | TestPressEntity](id="x", variant="v", entity=value)
        print(f"{label:28} -> accepted")
    except ValidationError:
        print(f"{label:28} -> refused")


## 7. What the built graph carries

One column for the field — a reader has to see that the row has a continuation at all — and one edge per alternative, which is where a consumer finds the table it can live in.


In [ ]:
machine = ActionProductMachine()
head_node = next(node for node in machine.graph_coordinator.get_all_nodes() if node.label == "VinylRecordEntity")

for edge in head_node.get_all_edges():
    if edge.edge_name == "entity_specialization":
        print(f"entity_specialization   {edge.properties['classifier_value']:8} -> {edge.target_node_id}")

columns = [e.target_node.label for e in head_node.get_all_edges() if e.edge_name == "entity_field"]
print("entity_field columns   :", [name for name in columns if name is not None])


## Summary

One head field, three extension tables, one classifier choosing among them. The framework reads the axis, the value carries the identifier, the variant and the row, a class outside the union is refused, and the graph carries both the column and the edges.

Next — **[Lifecycle](../step_22-lifecycle.md)**: an entity's status as a finite-state machine with verifiable transitions.
